In [9]:
"""
Self-contained noisy-simulator validation figures.
Input : noisy_sim_validation_results.json
Output: fig_ns1_F_comparison.pdf
        fig_ns2_psucc_comparison.pdf
        fig_ns3_bloch_comparison.pdf
        fig_ns4_bootstrap_overlay.pdf
        fig_ns_panel.pdf  (paper figure)
"""

import json, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import proj3d

# =============================================================================
#  CONFIG
# =============================================================================
NS_JSON = "/Users/nandan/Desktop/CTCs/IBM/noisy_sim_validation_results.json"
OUT_DIR = "./"

with open(NS_JSON) as f:
    D = json.load(f)

hw   = D['hardware']
sims = D['sim_results']
meta = D['metadata']
cal  = D['calibration']
shots = meta['shots']

# Ideal reference
sx_ideal=-0.2384; sy_ideal=0.5179; sz_ideal=-0.8215

# Display config: (key, short_label, color)
# Only include keys that exist in results
SIM_ORDER = ['NM0_noiseless','NM1_depol','NM2_thermal','NM3_full','NM4_auto']
CONFIGS = [
    ('NM0_noiseless', 'Noiseless',          '#27AE60'),
    ('NM1_depol',     'Depol only',          '#3498DB'),
    ('NM2_thermal',   'Depol+T1/T2',         '#9B59B6'),
    ('NM3_full',      'Full model',           '#E67E22'),
    ('NM4_auto',      'from_backend()',       '#C0392B'),
]
CONFIGS = [(k,l,c) for k,l,c in CONFIGS if k in sims]

C_HW    = '#2C3E50'
C_IDEAL = '#27AE60'
os.makedirs(OUT_DIR, exist_ok=True)

plt.rcParams.update({
    'font.family': 'DejaVu Serif', 'font.size': 11,
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'xtick.labelsize': 9.5, 'ytick.labelsize': 10,
    'legend.fontsize': 9, 'figure.dpi': 180,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.8, 'pdf.fonttype': 42,
})

# =============================================================================
#  FIG 1 — F_msg: hardware vs each noise model (horizontal bar)
# =============================================================================
def fig_F_comparison():
    fig, ax = plt.subplots(figsize=(7.5, 4.2))

    labels = [l for _,l,_ in CONFIGS] + ['Hardware\n(ibm_torino)']
    colors = [c for _,_,c in CONFIGS] + [C_HW]

    F_vals  = [sims[k]['F']    for k,_,_ in CONFIGS] + [hw['F']]
    F_elo   = [sims[k]['F']    - sims[k]['F_lo']  for k,_,_ in CONFIGS] + [hw['F']-hw['F_lo']]
    F_ehi   = [sims[k]['F_hi'] - sims[k]['F']     for k,_,_ in CONFIGS] + [hw['F_hi']-hw['F']]

    y = np.arange(len(labels))
    bars = ax.barh(y, F_vals, color=colors, height=0.55,
                   xerr=[F_elo, F_ehi],
                   error_kw=dict(elinewidth=1.5, capsize=5,
                                 capthick=1.5, ecolor='#333'),
                   zorder=3, edgecolor='white')
    bars[-1].set_hatch('///')   # hardware bar hatched

    # Hardware reference line
    ax.axvline(hw['F'], color=C_HW, ls='--', lw=1.5, alpha=0.7,
               label=f'Hardware $F={hw["F"]:.4f}$')
    ax.axvspan(hw['F_lo'], hw['F_hi'], alpha=0.08, color=C_HW,
               label='Hardware 95% CI')

    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=10)
    ax.set_xlabel('Fidelity $F(\\rho_Y,\\, \\rho_M)$')
    ax.set_title('Noisy-simulator validation: $F_{\\rm msg}$\n'
                 'Shaded region = hardware 95% CI ',
                 pad=8)
    ax.set_xlim(max(0.4, min(F_vals)-0.08), 1.05)
    ax.xaxis.grid(True, alpha=0.3, zorder=0)
    ax.legend(fontsize=9, framealpha=0.9, loc='upper right')

    # Annotate ΔF
    for yi, (k,l,c) in enumerate(CONFIGS):
        delta = sims[k]['delta_F']
        sign  = '+' if delta >= 0 else ''
        col   = C_IDEAL if abs(delta) < 0.05 else (
                '#E67E22' if abs(delta) < 0.10 else '#E74C3C')
        ax.text(sims[k]['F'] + sims[k]['F_hi'] - sims[k]['F'] + 0.005,
                yi, f'$\\Delta F={sign}{delta:.4f}$',
                va='center', fontsize=8.5, color=col)

    fig.tight_layout()
    return fig

# =============================================================================
#  FIG 2 — p_succ: hardware vs each noise model
# =============================================================================
def fig_psucc_comparison():
    fig, ax = plt.subplots(figsize=(7.5, 4.0))

    labels = [l for _,l,_ in CONFIGS] + ['Hardware\n(ibm_torino)']
    colors = [c for _,_,c in CONFIGS] + [C_HW]

    p_vals = [sims[k]['p_succ'] for k,_,_ in CONFIGS] + [hw['p_succ']]
    p_elo  = [sims[k]['p_succ'] - sims[k]['p_lo'] for k,_,_ in CONFIGS] + [hw['p_succ']-hw['p_lo']]
    p_ehi  = [sims[k]['p_hi']   - sims[k]['p_succ'] for k,_,_ in CONFIGS] + [hw['p_hi']-hw['p_succ']]

    y = np.arange(len(labels))
    bars = ax.barh(y, p_vals, color=colors, height=0.55,
                   xerr=[p_elo, p_ehi],
                   error_kw=dict(elinewidth=1.5, capsize=5,
                                 capthick=1.5, ecolor='#333'),
                   zorder=3, edgecolor='white')
    bars[-1].set_hatch('///')

    ax.axvline(0.25, color=C_IDEAL, ls=':', lw=1.4, alpha=0.7,
               label='Ideal $p_{\\rm succ}=0.25$')
    ax.axvline(hw['p_succ'], color=C_HW, ls='--', lw=1.5, alpha=0.7,
               label=f'Hardware $p_{{\\mathrm{{succ}}}}={hw["p_succ"]:.4f}$')
    ax.axvspan(hw['p_lo'], hw['p_hi'], alpha=0.08, color=C_HW,
               label='Hardware 95% CI')

    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=10)
    ax.set_xlabel('Post-selection rate $p_{\\rm succ}$')
    ax.set_title('Noisy-simulator validation: $p_{\\rm succ}$', pad=8)
    ax.set_xlim(0.15, 0.32)
    ax.xaxis.grid(True, alpha=0.3, zorder=0)
    ax.legend(fontsize=9, framealpha=0.9, loc='lower right')
    fig.tight_layout()
    return fig

# =============================================================================
#  FIG 3 — Bloch sphere: hardware vs noise models
# =============================================================================
class Arrow3D(FancyArrowPatch):
    def __init__(self, xs, ys, zs, *args, **kwargs):
        super().__init__((0,0),(0,0), *args, **kwargs)
        self._verts3d = xs, ys, zs
    def do_3d_projection(self, renderer=None):
        xs,ys,zs=proj3d.proj_transform(*self._verts3d,self.axes.M)
        self.set_positions((xs[0],ys[0]),(xs[1],ys[1]))
        return np.min(zs)

def draw_bloch(ax, vectors, labels, colors):
    u=np.linspace(0,2*np.pi,60); v=np.linspace(0,np.pi,40)
    ax.plot_wireframe(np.outer(np.cos(u),np.sin(v)),
                      np.outer(np.sin(u),np.sin(v)),
                      np.outer(np.ones(60),np.cos(v)),
                      rstride=5,cstride=5,color='#ccc',lw=0.3,alpha=0.35)
    for xyz,lbl in [([1.2,0,0],'$x$'),([0,1.2,0],'$y$'),([0,0,1.2],'$z$'),
                    ([-1.2,0,0],''),([ 0,-1.2,0],''),([ 0,0,-1.2],'')]:
        ax.plot([0,xyz[0]],[0,xyz[1]],[0,xyz[2]],color='#aaa',lw=0.7,alpha=0.7)
        if lbl: ax.text(xyz[0]*1.12,xyz[1]*1.12,xyz[2]*1.12,lbl,fontsize=9,ha='center',color='#555')
    t=np.linspace(0,2*np.pi,120)
    for c1,c2,c3 in [(np.cos(t),np.sin(t),np.zeros(120)),
                     (np.cos(t),np.zeros(120),np.sin(t)),
                     (np.zeros(120),np.cos(t),np.sin(t))]:
        ax.plot(c1,c2,c3,color='#bbb',lw=0.4,alpha=0.4)
    for (bx,by,bz),lbl,col in zip(vectors,labels,colors):
        arw=Arrow3D([0,bx],[0,by],[0,bz],mutation_scale=14,lw=2.2,
                    arrowstyle='-|>',color=col)
        ax.add_artist(arw)
        ax.text(bx*1.15,by*1.15,bz*1.15,lbl,fontsize=9,color=col,fontweight='bold')
    ax.set_xlim(-1.3,1.3); ax.set_ylim(-1.3,1.3); ax.set_zlim(-1.3,1.3)
    ax.set_box_aspect([1,1,1]); ax.axis('off')

def fig_bloch():
    fig = plt.figure(figsize=(6.0, 5.2))
    ax  = fig.add_subplot(111, projection='3d')

    vecs   = [[sx_ideal, sy_ideal, sz_ideal]]
    labels = ['$\\rho_M$ (ideal)']
    colors = [C_IDEAL]

    # Hardware
    vecs.append([hw.get('sx', sims[list(sims.keys())[0]]['bloch']['sx']),
                 hw.get('sy', sims[list(sims.keys())[0]]['bloch']['sy']),
                 hw.get('sz', sims[list(sims.keys())[0]]['bloch']['sz'])])
    labels.append('Hardware'); colors.append(C_HW)

    # Noise models
    for k,l,c in CONFIGS:
        b = sims[k]['bloch']
        vecs.append([b['sx'], b['sy'], b['sz']])
        labels.append(l); colors.append(c)

    draw_bloch(ax, vecs, labels, colors)
    patches = [mpatches.Patch(color=c,label=l)
               for c,l in zip(colors,labels)]
    ax.legend(handles=patches, loc='upper left',
              bbox_to_anchor=(-0.1,1.0), fontsize=8.5, framealpha=0.9)
    ax.set_title('Bloch sphere: hardware vs noise models', fontsize=11, pad=4)
    fig.tight_layout()
    return fig

# =============================================================================
#  FIG 4 — Bootstrap F overlay: hardware + sim models
# =============================================================================
def fig_bootstrap():
    fig, ax = plt.subplots(figsize=(7.0, 4.2))

    # Hardware bootstrap — approximate from CI using Gaussian model
    # Build approximate hw bootstrap from CI (Gaussian approximation)
    hw_std  = (hw['F_hi'] - hw['F_lo']) / (2 * 1.96)
    hw_bsF  = np.random.default_rng(0).normal(hw['F'], hw_std, 2000)
    hw_bsF  = np.clip(hw_bsF, 0, 1)
    ax.hist(hw_bsF, bins=40, color=C_HW, alpha=0.55,
            edgecolor='white', lw=0.4, zorder=3,
            label=f'Hardware ($F={hw["F"]:.4f}$)')
    ax.axvline(hw['F'], color=C_HW, lw=2.0, zorder=5)
    ax.axvspan(hw['F_lo'], hw['F_hi'], alpha=0.10, color=C_HW)

    for k,l,c in CONFIGS:
        bsF = np.array(sims[k]['bootstrap_F'])
        med = np.median(bsF)
        lo  = np.percentile(bsF, 2.5)
        hi  = np.percentile(bsF, 97.5)
        ax.hist(bsF, bins=40, color=c, alpha=0.55,
                edgecolor='white', lw=0.4, zorder=3,
                label=f'{l}  ($F={sims[k]["F"]:.4f}$)')
        ax.axvline(med, color=c, lw=1.6, zorder=4)
        ax.axvline(lo,  color=c, lw=0.9, ls='--', zorder=4)
        ax.axvline(hi,  color=c, lw=0.9, ls='--', zorder=4)

    ax.axvline(1.0, color=C_IDEAL, lw=1.5, ls=':', zorder=6, label='Ideal $F=1$')
    ax.set_xlabel('Fidelity $F(\\rho_Y,\\, \\rho_M)$')
    ax.set_ylabel('Bootstrap count')
    ax.set_title('Bootstrap $F$ distributions: hardware vs noise models', pad=8)
    ax.legend(framealpha=0.9, edgecolor='#ccc', fontsize=8.5,
          loc='upper left', bbox_to_anchor=(0.485, 1.0))
    ax.yaxis.grid(True, alpha=0.3, zorder=0)
    fig.tight_layout()
    return fig

# =============================================================================
#  FIG 5 — Combined panel (agreement summary + detail)
# =============================================================================
def fig_panel():
    fig = plt.figure(figsize=(15.5, 5.2))
    gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.38)

    keys   = [k for k,_,_ in CONFIGS]
    labels = [l for _,l,_ in CONFIGS]
    colors = [c for _,_,c in CONFIGS]
    x = np.arange(len(keys))

    # ── (a) F_msg bar chart ──────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0])
    F_vals = [sims[k]['F']    for k in keys]
    F_elo  = [sims[k]['F']    - sims[k]['F_lo']  for k in keys]
    F_ehi  = [sims[k]['F_hi'] - sims[k]['F']     for k in keys]
    bars = ax1.bar(x, F_vals, color=colors, width=0.55,
                   yerr=[F_elo,F_ehi],
                   error_kw=dict(elinewidth=1.4,capsize=5,capthick=1.4,ecolor='#222'),
                   zorder=3, edgecolor='white')
    ax1.axhline(hw['F'],  color=C_HW,    ls='--', lw=1.6, alpha=0.8,
                label=f'Hardware $F={hw["F"]:.4f}$')
    ax1.axhspan(hw['F_lo'], hw['F_hi'], alpha=0.10, color=C_HW,
                label='HW 95% CI')
    ax1.set_xticks(x); ax1.set_xticklabels(labels, fontsize=9, rotation=15, ha='right')
    ymin = max(0.4, min(F_vals)-0.08)
    ax1.set_ylim(ymin, 1.05); ax1.set_ylabel('$F(\\rho_Y,\\rho_M)$')
    ax1.set_title('(a) $F_{\\rm msg}$: sim vs hardware', fontsize=11)
    ax1.yaxis.grid(True, alpha=0.3, zorder=0)
    ax1.legend(fontsize=8.5, framealpha=0.9)
    for xi,fv in enumerate(F_vals):
        ax1.text(xi, fv+F_ehi[xi]+0.004, f'{fv:.4f}', ha='center', fontsize=8)
    for xi,k in enumerate(keys):
        d=sims[k]['delta_F']; s='+' if d>=0 else ''
        col='#27AE60' if abs(d)<0.05 else ('#E67E22' if abs(d)<0.10 else '#E74C3C')
        ax1.text(xi, ymin+0.005, f'{s}{d:.3f}', ha='center',
                 fontsize=7.5, color=col, style='italic')

    # ── (b) p_succ bar chart ─────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1])
    p_vals = [sims[k]['p_succ'] for k in keys]
    p_elo  = [sims[k]['p_succ'] - sims[k]['p_lo']  for k in keys]
    p_ehi  = [sims[k]['p_hi']   - sims[k]['p_succ'] for k in keys]
    ax2.bar(x, p_vals, color=colors, width=0.55,
            yerr=[p_elo,p_ehi],
            error_kw=dict(elinewidth=1.4,capsize=5,capthick=1.4,ecolor='#222'),
            zorder=3, edgecolor='white')
    ax2.axhline(hw['p_succ'], color=C_HW, ls='--', lw=1.6, alpha=0.8,
                label=f'Hardware $p_{{\\mathrm{{succ}}}}={hw["p_succ"]:.4f}$')
    ax2.axhspan(hw['p_lo'], hw['p_hi'], alpha=0.10, color=C_HW)
    ax2.axhline(0.25, color=C_IDEAL, ls=':', lw=1.2, alpha=0.7,
                label='Ideal 0.25')
    ax2.set_xticks(x); ax2.set_xticklabels(labels, fontsize=9, rotation=15, ha='right')
    ax2.set_ylim(0.15, 0.32); ax2.set_ylabel('$p_{\\rm succ}$')
    ax2.set_title('(b) $p_{\\rm succ}$: sim vs hardware', fontsize=11)
    ax2.yaxis.grid(True, alpha=0.3, zorder=0)
    ax2.legend(fontsize=8.5, framealpha=0.9)
    for xi,pv in enumerate(p_vals):
        ax2.text(xi, pv+p_ehi[xi]+0.003, f'{pv:.4f}',
                 ha='center', fontsize=8)

    # ── (c) |ΔF| and |Δp| agreement summary ─────────────────────────────────
    ax3 = fig.add_subplot(gs[2])
    dF_vals = [abs(sims[k]['delta_F']) for k in keys]
    dp_vals = [abs(sims[k]['delta_p']) for k in keys]
    w = 0.3; xd = np.arange(len(keys))
    b1 = ax3.bar(xd-w/2, dF_vals, width=w, color=colors,
                 edgecolor='white', zorder=3, label='$|\\Delta F|$')
    b2 = ax3.bar(xd+w/2, dp_vals, width=w, color=colors,
                 edgecolor='white', zorder=3, alpha=0.55,
                 hatch='///', label='$|\\Delta p_{\\rm succ}|$')
    ax3.axhline(0.05, color='#27AE60', ls='--', lw=1.2, alpha=0.7,
                label='5% threshold')
    ax3.set_xticks(xd); ax3.set_xticklabels(labels, fontsize=9, rotation=15, ha='right')
    ax3.set_ylabel('Absolute discrepancy vs hardware')
    ax3.set_title('(c) Agreement: $|\\Delta F|$ and $|\\Delta p_{\\rm succ}|$',
                  fontsize=11)
    ax3.yaxis.grid(True, alpha=0.3, zorder=0)
    ax3.legend(fontsize=9, framealpha=0.9)

    best = D['best_model']
    fig.suptitle(
        f'Noisy-simulator validation — ibm\\_torino  ({shots:,} shots/basis)\n'
        f'Hardware: $F={hw["F"]:.4f}$, $p_{{\\mathrm{{succ}}}}={hw["p_succ"]:.4f}$  |  '
        f'Best model: {sims[best]["label"]}  ($|\\Delta F|='
        f'{abs(sims[best]["delta_F"]):.4f}$)',
        fontsize=11, y=1.02
    )
    fig.tight_layout()
    return fig

# =============================================================================
#  Save all
# =============================================================================
N_BS = 2000   # used in fig_bootstrap approximation
print("Generating noisy-simulator validation figures ...")
for fname, fn in [
    ('fig_ns1_F_comparison.pdf',     fig_F_comparison),
    ('fig_ns2_psucc_comparison.pdf', fig_psucc_comparison),
    ('fig_ns3_bloch_comparison.pdf', fig_bloch),
    ('fig_ns4_bootstrap_overlay.pdf',fig_bootstrap),
    ('fig_ns_panel.pdf',             fig_panel),
]:
    try:
        f = fn()
        f.savefig(os.path.join(OUT_DIR, fname), bbox_inches='tight', dpi=200)
        plt.close(f)
        print(f"  [✓] {fname}")
    except Exception as e:
        print(f"  [!] {fname} failed: {e}")

print(f"\n{'='*60}")
print("  PAPER-READY AGREEMENT TABLE")
print(f"{'='*60}")
print(f"  Hardware: F={hw['F']:.4f}  p_succ={hw['p_succ']:.4f}\n")
print(f"  {'Model':26s}  {'|ΔF|':>8}  {'|Δp|':>8}  {'Agreement'}")
print("  " + "-" * 58)
for k,l,_ in CONFIGS:
    r = sims[k]
    aF = abs(r['delta_F']); ap = abs(r['delta_p'])
    agree = "good (<5%)" if aF<0.05 else ("ok (<10%)" if aF<0.10 else "poor")
    print(f"  {l:26s}  {aF:>8.4f}  {ap:>8.4f}  {agree}")
print(f"\n  Best model: {sims[D['best_model']]['label']}")
print(f"[✓] Done.")

Generating noisy-simulator validation figures ...
  [✓] fig_ns1_F_comparison.pdf
  [✓] fig_ns2_psucc_comparison.pdf
  [✓] fig_ns3_bloch_comparison.pdf
  [✓] fig_ns4_bootstrap_overlay.pdf
  [✓] fig_ns_panel.pdf

  PAPER-READY AGREEMENT TABLE
  Hardware: F=0.8577  p_succ=0.2430

  Model                           |ΔF|      |Δp|  Agreement
  ----------------------------------------------------------
  Noiseless                     0.1395    0.0028  poor
  Depol only                    0.1374    0.0089  poor
  Depol+T1/T2                   0.1377    0.0074  poor
  Full model                    0.1411    0.0064  poor
  from_backend()                0.0355    0.0070  good (<5%)

  Best model: AerSimulator.from_backend()
[✓] Done.


/var/folders/dg/bj9b922n29vcxp68247bp7vh0000gn/T/ipykernel_15510/187506829.py:343: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
